# Demo 09a — Mixtures and PCA

**Class 09 · B1** (slides: `slides/09_blind_signal_separation.md`)

The running example of the class: **two source signals mixed into two microphones**.
We look at the mixtures, then at PCA: what it finds (variance directions, not sources),
how a rank-1 reconstruction drops the second signal, and where the scree plot comes from.
The last sections move PCA to digits (components as images, reconstruction) and to spambase
(a scree over 57 features), with two failure cases no slide shows.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from faa import bss, data, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_width_chars(150)
plotlib.setup(module="09")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
SEED = 0


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line."""
    print(f"--- slide: {title} ---")


def show(frame: pl.DataFrame, digits: int = 3) -> None:
    print(frame.with_columns(pl.col(pl.Float64).round(digits)))

## The running example: two microphones, two sources

Two sines (3 Hz and a 7 Hz chirp) mixed by $A = \begin{pmatrix}0.9 & 0.4\\0.7 & 1.1\end{pmatrix}$,
sensor noise 0.05. This figure is the opening frame of the deck: sources on the left, mixtures on the right.

In [ ]:
X, S, A = bss.mixtures(seed=SEED)
n = X.shape[1]
t = np.arange(n) / n
f, ax = plt.subplots(2, 2, figsize=(plotlib.FULL, 2.3), sharex=True)
for j, (sig, col, name) in enumerate([(S[0], BLUE, "source 1 (3 Hz sine)"), (S[1], ORANGE, "source 2 (chirp)")]):
    ax[j, 0].plot(t, sig, color=col, lw=0.9)
    ax[j, 0].set_ylabel(f"$s_{j + 1}$")
    ax[j, 0].set_title(name, fontsize=7.5)
for j in range(2):
    ax[j, 1].plot(t, X[j], color=GREY, lw=0.9)
    ax[j, 1].set_ylabel(f"$x_{j + 1}$")
ax[0, 1].set_title("microphone 1: 0.9 s₁ + 0.4 s₂", fontsize=7.5)
ax[1, 1].set_title("microphone 2: 0.7 s₁ + 1.1 s₂", fontsize=7.5)
ax[1, 0].set_xlabel("time (s)")
ax[1, 1].set_xlabel("time (s)")
plotlib.save(f, "p1_mixtures")
print(f"A =\n{A}")
print("correlation of x1, x2:", f"{np.corrcoef(X)[0, 1]:.2f}")

## Slide: What PCA finds on the mixtures

PCA on the 2-D cloud of microphone samples finds the **variance directions**, not the sources.
The recovered signals are the projections; their correlation with the truth is far from 1.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2).fit(X.T)
P = pca.components_  # rows are the PCs, in the (samples x features) convention of X.T
f, ax = plt.subplots(figsize=(plotlib.COL, 2.1))
ax.scatter(X[0], X[1], s=3, color=GREY, alpha=0.5)
for j, (v, col) in enumerate([(pca.components_[0], BLUE), (pca.components_[1], ORANGE)]):
    ax.annotate("", xy=(v * 1.8), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=col, lw=1.6))
    ax.text(v[0] * 2.0, v[1] * 2.0, f"PC{j + 1}", color=col, fontsize=7, ha="center")
ax.set_xlabel("microphone 1")
ax.set_ylabel("microphone 2")
plotlib.save(f, "p1_pca_axes")
print(f"explained variance ratio: {pca.explained_variance_ratio_.round(3)}")
print(f"PC axes (rows):\n{pca.components_.round(3)}")
print(f"true mixing A (columns are the source directions):\n{A.round(3)}")

## Slide: PCA separates the mixtures badly

The PC projections, matched to the true sources: correlations around 0.5-0.8, not the sources.
(The slide quotes these numbers; demo 09b shows ICA fixing them.)

In [ ]:
P = pca.transform(X.T).T
matched, corr = bss.match(S, P)
f, ax = plt.subplots(2, 2, figsize=(plotlib.FULL, 2.3), sharex=True)
for j in range(2):
    ax[j, 0].plot(t, S[j], color=(BLUE, ORANGE)[j], lw=0.9)
    ax[j, 0].set_ylabel(f"$s_{j + 1}$ true")
    ax[j, 1].plot(t, matched[j], color=GREY, lw=0.9)
    ax[j, 1].set_ylabel("PC proj.")
    ax[j, 1].set_title(f"corr {corr[j]:.2f}", fontsize=7.5)
ax[1, 0].set_xlabel("time (s)")
ax[1, 1].set_xlabel("time (s)")
plotlib.save(f, "p1_pca_separation")
print(f"correlation with truth: {corr.round(3)}")
print(f"SNR: {bss.snr_db(S, P).round(1)} dB")

## Slide: Rank-1 reconstruction drops a source

PCA reconstruction with 1 component = the data projected on the biggest variance direction:
the second source is gone from microphone 1 but leaks into the reconstruction.

In [ ]:
pca1 = PCA(n_components=1).fit(X.T)
X1 = pca1.inverse_transform(pca1.transform(X.T)).T
f, ax = plt.subplots(2, 2, figsize=(plotlib.FULL, 2.3), sharex=True)
for j in range(2):
    ax[j, 0].plot(t, X[j], color=GREY, lw=0.9)
    ax[j, 0].set_ylabel(f"$x_{j + 1}$")
    ax[j, 1].plot(t, X1[j], color=BLUE, lw=0.9)
    ax[j, 1].set_ylabel("1 PC")
ax[0, 0].set_title("microphone 1", fontsize=7.5)
ax[0, 1].set_title("reconstruction", fontsize=7.5)
ax[1, 0].set_xlabel("time (s)")
ax[1, 1].set_xlabel("time (s)")
plotlib.save(f, "p1_rank1")
print(f"relative reconstruction error (1 PC): {np.linalg.norm(X - X1) / np.linalg.norm(X):.3f}")
print(
    f"relative error (2 PCs): {np.linalg.norm(X - PCA(2).fit(X.T).inverse_transform(PCA(2).fit(X.T).transform(X.T)).T) / np.linalg.norm(X):.3f}"
)

## Slide: The scree plot decides how many components

PCA on spambase (57 features, standardised): cumulative explained variance and the scree.
90 % of the variance sits in 30 of 57 components; the knee is a decision, not a law.

In [ ]:
Xspam, _, names = data.load_table("spambase", "spam")
mu, sd = Xspam.mean(0), Xspam.std(0)
Xs = (Xspam - mu) / sd
pca_s = PCA().fit(Xs)
evr = pca_s.explained_variance_ratio_
f, ax = plt.subplots(figsize=(plotlib.COL, 2.1))
ax.plot(np.arange(1, 58), evr, color=BLUE, lw=1.2)
ax.plot(np.arange(1, 58), np.cumsum(evr), color=ORANGE, lw=1.2)
ax.set_xlabel("component")
ax.set_ylabel("variance share")
ax.legend(["individual", "cumulative"], fontsize=7)
plotlib.save(f, "p1_scree")
k90 = int(np.searchsorted(np.cumsum(evr), 0.9) + 1)
print(f"components for 90 % variance: {k90}")
print(f"first three shares: {evr[:3].round(3)}")

## Slide: Components of digits are averages of digits

PCA on digits (8 x 8 images): the mean image, PC1-PC4 and the rank-r reconstruction of one digit.
Components look like blurry digits, not like parts.

In [ ]:
Xd, _, _ = data.load_table("digits", "target")
Xd = Xd / Xd.max()
pca_d = PCA().fit(Xd)
digit = Xd[0]
f, ax = plt.subplots(2, 4, figsize=(plotlib.FULL, 2.7))
ims = [digit.reshape(8, 8)] + [pca_d.mean_.reshape(8, 8)] + [pca_d.components_[j].reshape(8, 8) for j in range(2)]
titles = ["a digit", "mean image", "PC1", "PC2"]
for a, im, ti in zip(ax[0], ims, titles):
    a.imshow(im, cmap="viridis")
    a.set_title(ti, fontsize=7.5)
    a.set_xticks([])
    a.set_yticks([])
    a.set_aspect("equal")
for a, r in zip(ax[1], [16, 8, 4]):
    rec = pca_d.mean_ + (pca_d.transform(digit[None]) @ pca_d.components_[:, :r] @ pca_d.components_[:r])[:, :64][0]
    a.imshow(rec.reshape(8, 8), cmap="viridis")
    a.set_title(f"{r} components", fontsize=7.5)
    a.set_xticks([])
    a.set_yticks([])
    a.set_aspect("equal")
ax[1, 3].axis("off")
plotlib.save(f, "p1_digits")

## Slide: SVD is the basis of PCA

Data matrix in the deck convention: columns are examples, $X_c$ is $d \times n$ and centred.
$X_c = U \Sigma V^\top = \sum_j \sigma_j u_j v_j^\top$; the eigenvalues of
$C = X_c X_c^\top/(n-1)$ are $\sigma_j^2/(n-1)$ and the loadings are the columns of $U$.

In [ ]:
Xc = (Xd - Xd.mean(0)).T  # 64 x n, the deck convention (examples are columns)
U, sv, Vt = np.linalg.svd(Xc, full_matrices=False)
n_ex = Xc.shape[1]
lam = sv**2 / (n_ex - 1)
slide("SVD and PCA agree")
print(f"X_c shape {Xc.shape}; U {U.shape}, sigma {sv.shape}, V^T {Vt.shape}")
print(f"largest |lambda_j - sklearn explained_variance_| over 64: {np.abs(lam - pca_d.explained_variance_).max():.1e}")
assert np.allclose(lam, pca_d.explained_variance_, atol=1e-8)
print(f"|u_1 . sklearn PC1| = {abs(U[:, 0] @ pca_d.components_[0]):.3f}")
print(f"singular values 1-4: {sv[:4].round(1)}")

i0 = 0  # the digit used in the figures
z = sv * Vt[:, i0]  # coefficients of digit i0 on u_1 .. u_64: sigma_j v_j[i]
k_show = 5
f, ax = plt.subplots(2, k_show, figsize=(plotlib.FULL, 2.3))
for j in range(k_show):
    lim = np.abs(U[:, j]).max()
    ax[0, j].imshow(U[:, j].reshape(8, 8), cmap="RdBu_r", vmin=-lim, vmax=lim)
    ax[0, j].set_title(f"$u_{j + 1}$, $\\sigma_{j + 1}$={sv[j]:.0f}", fontsize=8)
    term = np.outer(U[:, j], Vt[j, i0]) * sv[j]
    lim2 = np.abs(Xc[:, i0]).max()
    ax[1, j].imshow(term.reshape(8, 8), cmap="RdBu_r", vmin=-lim2, vmax=lim2)
    ax[1, j].set_title(f"{z[j]:+.2f} $u_{j + 1}$", fontsize=8)
    for r in range(2):
        ax[r, j].set_xticks([])
        ax[r, j].set_yticks([])
        ax[r, j].set_aspect("equal")
plotlib.save(f, "p1_svd_terms")
print(f"digit {i0}: first five coefficients {z[:5].round(2)}")

## Slide: Fewer singular values, larger error

Keeping the first $k$ terms is the best rank-$k$ approximation (Eckart–Young); the squared
Frobenius error is exactly the sum of the discarded $\sigma_j^2$.

In [ ]:
ks = [1, 2, 4, 8, 16, 64]
digits_show = [0]
f = plt.figure(figsize=(plotlib.FULL, 1.9))
gs = f.add_gridspec(2, len(ks) + 1, height_ratios=[1, 1.5], hspace=0.25)
for r, i in enumerate(digits_show):
    a = f.add_subplot(gs[r, 0])
    a.imshow((Xc[:, i] + Xd.mean(0)).reshape(8, 8), cmap="viridis")
    a.set_title("digit" if r == 0 else "", fontsize=8)
    a.set_xticks([]), a.set_yticks([])
    for c, k in enumerate(ks):
        rec = (U[:, :k] * sv[:k]) @ Vt[:k, i] + Xd.mean(0)
        a = f.add_subplot(gs[r, c + 1])
        a.imshow(rec.reshape(8, 8), cmap="viridis", vmin=0, vmax=1)
        if r == 0:
            a.set_title(f"$k$={k}", fontsize=8)
        a.set_xticks([]), a.set_yticks([])
ax = f.add_subplot(gs[1, :])
ks_all = np.arange(0, 65)
tail = np.sqrt(np.maximum(0, np.cumsum((sv**2)[::-1])[::-1]))  # sqrt(sum_{j>=k} sigma_j^2) for k=0..63
err_formula = np.append(tail, 0.0) / np.linalg.norm(Xc)
err_direct = np.array([np.linalg.norm(Xc - (U[:, :k] * sv[:k]) @ Vt[:k]) / np.linalg.norm(Xc) for k in ks_all])
ax.plot(ks_all, err_direct, color=BLUE, lw=1.4, label="measured")
ax.plot(ks_all[::4], err_formula[::4], "o", color=ORANGE, ms=3, label=r"$\sqrt{\sum_{j>k}\sigma_j^2}/\|X\|$")
ax.set_xlabel("terms kept $k$")
ax.set_ylabel("relative error")
ax.legend(fontsize=8.5)
plotlib.save(f, "p1_svd_recon")
assert np.allclose(err_direct, err_formula, atol=1e-8)
slide("Eckart-Young check")
for k in ks:
    print(f"k={k:2d}: relative error {err_direct[k]:.3f}, variance kept {np.cumsum(lam)[k - 1] / lam.sum():.3f}")

## Slide: PCA by hand, ten points

The classic ten-point example: centre, covariance, eigen-decomposition, project, rebuild.
Every number of the "PCA by Hand" frame is printed here.

In [ ]:
pts = np.array(
    [
        [2.5, 2.4],
        [0.5, 0.7],
        [2.2, 2.9],
        [1.9, 2.2],
        [3.1, 3.0],
        [2.3, 2.7],
        [2.0, 1.6],
        [1.0, 1.1],
        [1.5, 1.6],
        [1.1, 0.9],
    ]
)
Xh = pts.T  # 2 x 10, columns are examples
mu_h = Xh.mean(axis=1, keepdims=True)
Xhc = Xh - mu_h
Ch = Xhc @ Xhc.T / (Xh.shape[1] - 1)
lam_h, W_h = np.linalg.eigh(Ch)
lam_h, W_h = lam_h[::-1], W_h[:, ::-1]
if W_h[0, 0] < 0:
    W_h[:, 0] = -W_h[:, 0]
Z_h = W_h.T @ Xhc
X1_h = W_h[:, :1] @ Z_h[:1]
slide("PCA by hand")
print(f"1 mean: {mu_h.ravel().round(2)}")
print(f"2 covariance: var1 {Ch[0, 0]:.3f}, var2 {Ch[1, 1]:.3f}, cov {Ch[0, 1]:.3f}")
print(f"3 eigenvalues: {lam_h.round(3)}; PC1 = {W_h[:, 0].round(3)}, PC2 = {W_h[:, 1].round(3)}")
print(f"4 share of PC1: {lam_h[0] / lam_h.sum():.3f}")
print(f"5 scores of the first three points on PC1: {Z_h[0, :3].round(3)}")
print(f"6 rank-1 rebuild error (Frobenius, relative): {np.linalg.norm(Xhc - X1_h) / np.linalg.norm(Xhc):.3f}")
print(
    f"check: error^2 / ((n-1) trace) = {np.linalg.norm(Xhc - X1_h) ** 2 / ((Xh.shape[1] - 1) * lam_h.sum()):.3f} equals the share of PC2 {lam_h[1] / lam_h.sum():.3f}"
)
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.7))
ax[0].scatter(*Xhc, color=GREY, s=14)
for j, col in enumerate([BLUE, ORANGE]):
    v = W_h[:, j] * 1.6
    ax[0].annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="->", color=col, lw=1.5))
    ax[0].text(v[0] * 1.08, v[1] * 1.08, f"PC{j + 1}", color=col, fontsize=8)
ax[0].set_xlabel("$x_1$ centred", fontsize=8)
ax[0].set_ylabel("$x_2$ centred", fontsize=8)
ax[0].set_title("centred points and axes", fontsize=8, pad=10)
ax[0].set_aspect("equal")
ax[1].scatter(*Xhc, color=GREY, s=14, alpha=0.5)
ax[1].scatter(*X1_h, color=BLUE, s=14)
for a, b in zip(Xhc.T, X1_h.T):
    ax[1].plot([a[0], b[0]], [a[1], b[1]], color=RED, lw=0.7)
ax[1].set_xlabel("$x_1$ centred", fontsize=8)
ax[1].set_title("rank-1 rebuild (red: error)", fontsize=8)
ax[1].set_aspect("equal")
plotlib.save(f, "p1_pca_hand")

## Slide: PCA, regression and autoencoders

Regression minimises vertical distances, PCA the orthogonal ones; a linear autoencoder with
a bottleneck of c units spans the same subspace as c principal components.

In [ ]:
import os

from scipy.linalg import subspace_angles

os.environ.setdefault("KERAS_BACKEND", "jax")
import keras

Xm, _, _ = bss.mixtures(seed=SEED)
Xmc = Xm - Xm.mean(axis=1, keepdims=True)
cov_m = Xmc @ Xmc.T / (Xm.shape[1] - 1)
w1 = np.linalg.eigh(cov_m)[1][:, -1]
b_yx = cov_m[0, 1] / cov_m[0, 0]  # regress x2 on x1: vertical residuals
b_xy = cov_m[1, 1] / cov_m[0, 1]  # regress x1 on x2: horizontal residuals, as a slope of x2 on x1
ang = {
    "x2 on x1": np.degrees(np.arctan(b_yx)),
    "x1 on x2": np.degrees(np.arctan(b_xy)),
    "PC1": np.degrees(np.arctan2(w1[1], w1[0])) % 180,
}


def orth_sse(direction):
    d = np.asarray(direction) / np.linalg.norm(direction)
    return float(np.sum((Xmc - np.outer(d, d) @ Xmc) ** 2))


sse = {"x2 on x1": orth_sse([1, b_yx]), "x1 on x2": orth_sse([1, b_xy]), "PC1": orth_sse(w1)}
slide("Three lines through one cloud")
print(f"angles (deg): { ({k: round(float(v), 1) for k, v in ang.items()}) }")
print(f"sum of squared orthogonal distances: { ({k: round(v, 1) for k, v in sse.items()}) }")
assert sse["PC1"] <= min(sse["x2 on x1"], sse["x1 on x2"])
f, ax = plt.subplots(figsize=(plotlib.FULL, 1.9))
ax.scatter(*Xmc, color=GREY, s=4, alpha=0.5)
tt = np.linspace(-2.2, 2.2, 2)
ax.plot(tt, b_yx * tt, color=BLUE, lw=1.3, label="regress $x_2$ on $x_1$")
ax.plot(tt, b_xy * tt, color=ORANGE, lw=1.3, label="regress $x_1$ on $x_2$")
ax.plot(tt, w1[1] / w1[0] * tt, color=GREEN, lw=1.6, label="PC1")
ax.set_xlim(-2.2, 2.2)
ax.set_ylim(-2.4, 2.4)
ax.set_xlabel("microphone 1 (centred)", fontsize=8)
ax.set_ylabel("microphone 2 (centred)", fontsize=8)
ax.legend(fontsize=8, loc="upper left")
plotlib.save(f, "p1_pca_vs_reg")

Xdc = Xd - Xd.mean(0)  # digits, examples as rows (keras convention)
keras.utils.set_random_seed(SEED)
ae = keras.Sequential(
    [keras.Input((64,)), keras.layers.Dense(8, use_bias=False), keras.layers.Dense(64, use_bias=False)]
)
ae.compile(optimizer=keras.optimizers.Adam(0.01), loss="mse")
ae.fit(Xdc, Xdc, epochs=150, batch_size=256, verbose=0)
rec_ae = ae.predict(Xdc, verbose=0)
pca8 = PCA(8).fit(Xd)
rec_pca = pca8.inverse_transform(pca8.transform(Xd)) - Xd.mean(0)
dec = ae.layers[-1].get_weights()[0]  # 8 x 64: the rows span the autoencoder subspace
slide("Linear autoencoder against PCA, 8 units")
print(f"reconstruction MSE: autoencoder {np.mean((Xdc - rec_ae) ** 2):.5f}, PCA {np.mean((Xdc - rec_pca) ** 2):.5f}")
print(
    f"largest principal angle between the two subspaces: {np.degrees(subspace_angles(dec.T, pca8.components_.T).max()):.1f} deg"
)
assert np.mean((Xdc - rec_ae) ** 2) < 1.15 * np.mean((Xdc - rec_pca) ** 2)

## Sweep and failure cases (more than the slides)

1. Scaling matters: PCA on raw spambase puts almost all variance in the huge-count features.
2. PCA on a Gaussian cloud: every direction is equal, the scree is flat, nothing separates.

In [ ]:
slide("Scaling failure")
pca_raw = PCA().fit(Xspam)
print(f"raw features, PC1 share: {pca_raw.explained_variance_ratio_[0]:.3f}")
print(f"standardised, PC1 share: {PCA().fit(Xs).explained_variance_ratio_[0]:.3f}")

slide("Gaussian cloud, flat scree")
g = np.random.default_rng(SEED).normal(size=(500, 6))
evr_g = PCA().fit(g).explained_variance_ratio_
print(f"explained variance ratio: {evr_g.round(3)}")

In [ ]:
print(f"total: {pl.DataFrame({'cell': ['A'], 's': [0]}).height and ''}demo 09a done")